# AuthentiHire — Phase 1: Exploratory Data Analysis & Preprocessing Pipeline
**Project:** AuthentiHire — Machine Learning System for Fraudulent Job & Internship Detection

**Scope of Phase 1:**
1. Inspect the raw `fake_job_postings.csv` dataset and summarize structural characteristics.
2. Evaluate missingness, duplicate entries, and class imbalance.
3. Identify high-signal vs low-signal / leaking features.
4. Implement a modular text cleaning and feature engineering pipeline.
5. Construct a strictly leakage-free stratified train/test split with TF-IDF vectorization.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import html
import re

from src.preprocessing import (
    clean_dataframe,
    prepare_train_test_splits,
    JobPostingPreprocessor,
    TEXT_COLUMNS,
    CATEGORICAL_COLUMNS
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)
print('Libraries and custom modules loaded.')

## 1. Dataset Dimensions & Schema Inspection

In [ ]:
DATA_PATH = '../data/fake_job_postings.csv'
df_raw = pd.read_csv(DATA_PATH)

print(f'Total Rows: {df_raw.shape[0]:,}')
print(f'Total Columns: {df_raw.shape[1]}')

summary_df = pd.DataFrame({
    'Dtype': df_raw.dtypes,
    'Non-Null Count': df_raw.notnull().sum(),
    'Null Count': df_raw.isnull().sum(),
    'Null Pct (%)': (df_raw.isnull().mean() * 100).round(2),
    'Unique Values': df_raw.nunique()
})
summary_df

## 2. Missing Value Analysis

In [ ]:
missing_counts = df_raw.isnull().sum().loc[lambda x: x > 0].sort_values(ascending=False)
missing_pcts = (missing_counts / len(df_raw)) * 100

fig, ax = plt.subplots(figsize=(10, 5))
missing_pcts.plot(kind='bar', color='#e74c3c', edgecolor='black', ax=ax)
ax.set_title('Missing Value Percentage by Feature', fontsize=13, fontweight='bold')
ax.set_ylabel('Percentage Missing (%)', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

for col, count in missing_counts.items():
    print(f'{col:25s}: {count:5d} missing ({missing_pcts[col]:.2f}%)')

## 3. Duplicate Rows Analysis

In [ ]:
exact_dups = df_raw.duplicated().sum()
content_dups = df_raw.drop(columns=['job_id']).duplicated().sum()
dup_groups = df_raw[df_raw.drop(columns=['job_id']).duplicated(keep=False)]

print(f'Exact full duplicates (including job_id): {exact_dups}')
print(f'Content duplicates (excluding job_id):    {content_dups}')
print(f'Total rows across duplicate clusters:    {len(dup_groups)}')
print('\nDuplicate breakdown by target (fraudulent):')
print(dup_groups['fraudulent'].value_counts())

## 4. Target Class Distribution & Imbalance

In [ ]:
target_counts = df_raw['fraudulent'].value_counts()
target_pcts = df_raw['fraudulent'].value_counts(normalize=True) * 100

print('=== TARGET DISTRIBUTION ===')
print(f'Legitimate (0): {target_counts[0]:,} ({target_pcts[0]:.2f}%)')
print(f'Fraudulent (1): {target_counts[1]:,} ({target_pcts[1]:.2f}%)')
print(f'Imbalance Ratio: {target_counts[0] / target_counts[1]:.2f} : 1')

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', color=['#2980b9', '#c0392b'], edgecolor='black', ax=ax)
ax.set_title('Target Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Postings')
ax.set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'], rotation=0)
plt.tight_layout()
plt.show()

## 5. Bivariate Analysis: Crucial Discriminative Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Company Logo
logo_ct = pd.crosstab(df_raw['has_company_logo'], df_raw['fraudulent'], normalize='columns') * 100
logo_ct.plot(kind='bar', ax=axes[0, 0], color=['#3498db', '#e67e22'], edgecolor='black')
axes[0, 0].set_title('Company Logo Presence', fontweight='bold')
axes[0, 0].set_ylabel('Percentage Within Class (%)')
axes[0, 0].set_xticklabels(['No Logo (0)', 'Has Logo (1)'], rotation=0)

# 2. Company Profile Presence
df_raw['has_profile'] = df_raw['company_profile'].notnull().astype(int)
prof_ct = pd.crosstab(df_raw['has_profile'], df_raw['fraudulent'], normalize='columns') * 100
prof_ct.plot(kind='bar', ax=axes[0, 1], color=['#3498db', '#e67e22'], edgecolor='black')
axes[0, 1].set_title('Company Profile Presence', fontweight='bold')
axes[0, 1].set_ylabel('Percentage Within Class (%)')
axes[0, 1].set_xticklabels(['No Profile (0)', 'Has Profile (1)'], rotation=0)

# 3. Has Questions
q_ct = pd.crosstab(df_raw['has_questions'], df_raw['fraudulent'], normalize='columns') * 100
q_ct.plot(kind='bar', ax=axes[1, 0], color=['#3498db', '#e67e22'], edgecolor='black')
axes[1, 0].set_title('Screening Questions Presence', fontweight='bold')
axes[1, 0].set_ylabel('Percentage Within Class (%)')
axes[1, 0].set_xticklabels(['No Questions (0)', 'Has Questions (1)'], rotation=0)

# 4. Salary Range Presence
df_raw['has_salary'] = df_raw['salary_range'].notnull().astype(int)
sal_ct = pd.crosstab(df_raw['has_salary'], df_raw['fraudulent'], normalize='columns') * 100
sal_ct.plot(kind='bar', ax=axes[1, 1], color=['#3498db', '#e67e22'], edgecolor='black')
axes[1, 1].set_title('Salary Range Stated', fontweight='bold')
axes[1, 1].set_ylabel('Percentage Within Class (%)')
axes[1, 1].set_xticklabels(['No Salary (0)', 'Has Salary (1)'], rotation=0)

plt.tight_layout()
plt.show()

## 6. Text Length & Linguistic Profile

In [ ]:
text_fields = ['title', 'company_profile', 'description', 'requirements', 'benefits']
length_data = []

for f in text_fields:
    r_len = df_raw[df_raw['fraudulent'] == 0][f].fillna('').apply(len)
    f_len = df_raw[df_raw['fraudulent'] == 1][f].fillna('').apply(len)
    length_data.append({
        'Field': f,
        'Real Mean Chars': round(r_len.mean(), 1),
        'Real Median Chars': round(r_len.median(), 1),
        'Fake Mean Chars': round(f_len.mean(), 1),
        'Fake Median Chars': round(f_len.median(), 1)
    })

pd.DataFrame(length_data)

## 7. Leakage-Free Preprocessing Pipeline & Stratified Split

In [ ]:
# Execute modular pipeline from src/preprocessing.py
splits = prepare_train_test_splits(DATA_PATH, test_size=0.2, random_state=42, drop_duplicates=True)

X_train = splits['X_train']
X_test = splits['X_test']
y_train = splits['y_train']
y_test = splits['y_test']
preprocessor = splits['preprocessor']

print(f'Train instances: {X_train.shape[0]:,} (Class 0: {(y_train == 0).sum():,}, Class 1: {(y_train == 1).sum():,})')
print(f'Test instances:  {X_test.shape[0]:,} (Class 0: {(y_test == 0).sum():,}, Class 1: {(y_test == 1).sum():,})')

X_train_tfidf, X_train_meta = preprocessor.transform(X_train)
X_test_tfidf, X_test_meta = preprocessor.transform(X_test)

print(f'\nTF-IDF Train Matrix Shape: {X_train_tfidf.shape}')
print(f'TF-IDF Test Matrix Shape:  {X_test_tfidf.shape}')
print(f'Engineered Metadata Features: {X_train_meta.shape[1]} columns')
print('\nSample extracted n-grams:', preprocessor.feature_names_[:20])